# Treinamento CatBoost

In [5]:
from pathlib import Path
import json
import pandas as pd
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


repo_root = find_repo_root(Path.cwd())
output_dir = repo_root / "src" / "prf_accidents_feature_importance" / "modeling" / "catboost" / "model"
output_dir.mkdir(parents=True, exist_ok=True)
train_dir = output_dir / "catboost_info"
train_dir.mkdir(parents=True, exist_ok=True)

parquet_path = repo_root / "data" / "processed" / "dummy_input.parquet"

print("Repositório detectado:", repo_root)
print("Parquet esperado:", parquet_path)
print("Diretório de artefatos CatBoost:", train_dir)

if not parquet_path.exists():
    print("O arquivo dummy_input.parquet não foi encontrado. Rode primeiro o notebook de geração de dados.")
else:
    # 1. Leitura de Dados
    df = pd.read_parquet(parquet_path)

    # 2. Limpeza de Nomenclatura (Garantindo que o erro de espaço não quebre o pipeline)
    df.columns = [column.strip() for column in df.columns]

    # 3. CONTRATO DE DADOS: Tipagem Estática Imutável
    categorical_cols = ["dia_semana", "horario", "condicao_metereologica", "tipo_pista", "reta", "inclinacao"]
    target_categorical_cols = ["classificacao_acidente"]
    numerical_cols = ["volume_pedagio", "icm_via"]

    # Forçando os tipos exatos antes de qualquer operação. O CatBoost adora strings nativas.
    for col in categorical_cols + target_categorical_cols:
        df[col] = df[col].astype(str)

    for col in numerical_cols:
        df[col] = df[col].astype(float)

    feature_columns = categorical_cols + numerical_cols
    target_column = "classificacao_acidente"

    X = df[feature_columns]
    y = df[target_column]

    # cat_features amarrado estritamente à regra de negócio, não à inferência do pandas
    cat_features = categorical_cols

    # 4. Divisão de Dados
    X_train, X_valid, y_train, y_valid = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # 5. Modelagem
    model = CatBoostClassifier(
        iterations=50,  # (Número de Árvores): O CatBoost é um algoritmo do tipo Ensemble que constrói várias árvores de decisão em sequência, onde uma tenta corrigir o erro da outra. 
                        # Esse parâmetro define que o modelo vai criar no máximo 50 árvores. É um valor bem baixo (geralmente usamos entre 500 e 1000 para a base final), 
                        # mas é excelente para rodar rápido durante essa fase de testes com os dados mockados.
        depth=4,        # (Profundidade da Árvore): Controla a complexidade de cada árvore individual. Uma profundidade de 4 significa que o modelo pode fazer até 4 perguntas em sequência 
                        # (ex: É pista simples? -> Está chovendo? -> É curva? -> O ICM é ruim?) antes de dar a resposta final. Cada árvore terá no máximo $2^4 = 16$ folhas.
                        # O valor 4 é conservador e ajuda a evitar o overfitting (quando o modelo "decora" os dados em vez de aprender padrões reais).
        learning_rate=0.1,  # (Taxa de Aprendizado): Define o "tamanho do passo" que o modelo dá a cada nova iteração. 
                            # Um valor de 0.1 (10%) significa que o peso da correção de cada nova árvore será multiplicado por 0.1 antes de ser adicionado ao modelo final.
                            # Regra de bolso: Se você aumentar as iterations, geralmente precisa diminuir o learning_rate (ex: 0.01 ou 0.05) para o modelo aprender mais devagar e com mais precisão.
        loss_function='MultiClass', # (Função de Perda): É a fórmula matemática que o algoritmo tenta minimizar (reduzir o erro) durante o treinamento.
                                    # Como a sua variável alvo (classificacao_acidente) possui três níveis categóricos (0 - Sem vítimas, 1 - Leves, 2 - Graves/Fatais), 
                                    # a função MultiClass diz ao algoritmo para calcular a probabilidade cruzada entre essas três categorias. Se fosse apenas binário (0 ou 1), usaríamos Logloss.
        auto_class_weights='Balanced', # Aplica um multiplicador no erro da classe minoritária: prestar mais atenção a ela durante o treinamento. Isso é útil quando temos classes desbalanceadas 
                                       # (ex: 90% da base é "Sem vítimas" e apenas 10% é "Graves/Fatais"). Isso despenca a Acurácia global, mas aumenta o F1-Score (Recall) das classes graves.
        eval_metric='TotalF1', # (Indicador de Desempenho): O F1 Score é uma métrica que combina precisão e recall em um único número. Ele é mais útil do que a acurácia em casos de classes desbalanceadas, 
                               # pois penaliza mais os falsos negativos e falsos positivos.
        verbose=False, # (Silenciar Saída): Por padrão, o CatBoost imprime no terminal o valor do erro a cada iteração (ex: Iteração 1: erro X... Iteração 50: erro Y).
                      # O "False" desliga esse comportamento, mantendo o notebook limpo e imprimindo apenas o resultado final que programado.
        train_dir=str(train_dir),
    )

    # Treinamento explícito
    model.fit(X_train, y_train, cat_features=cat_features)

    # 6. Avaliação e Exportação
    accuracy = model.score(X_valid, y_valid)
    # cerca de 90% dos acidentes são causados por fator humano (imprudência, álcool, excesso de velocidade). 
    # O modelo de está tentando prever a severidade de um acidente usando quase que exclusivamente infraestrutura e ambiente 
    # (pedágio, geometria, ICM). O algoritmo está tentando resolver um quebra-cabeça faltando 90% das peças que explicam a variabilidade do problema. 
    # Com o sinal fraco, a confiança nas previsões cai.
    # accuracy ~ 0.45 para os dados dummy -> acurácia baixa, porém, melhor do que chutes aleatórios (0.33).
    
    summary = {
        "model_name": "catboost_classifier",
        "feature_columns": feature_columns,
        "target_column": target_column,
        "n_samples": len(df),
        "accuracy": round(float(accuracy), 4),
        "status": "training completed",
    }

    output_path = output_dir / "model_summary.json"
    output_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

    print(f"Treinamento concluído com sucesso. Acurácia: {accuracy:.4f}")


Repositório detectado: c:\Users\augus\Desktop\UFSC\PPGESE\Disciplinas\Ciencias de Dados\prf-accidents-feature-importance
Parquet esperado: c:\Users\augus\Desktop\UFSC\PPGESE\Disciplinas\Ciencias de Dados\prf-accidents-feature-importance\data\processed\dummy_input.parquet
Diretório de artefatos CatBoost: c:\Users\augus\Desktop\UFSC\PPGESE\Disciplinas\Ciencias de Dados\prf-accidents-feature-importance\src\prf_accidents_feature_importance\modeling\catboost\model\catboost_info
Treinamento concluído com sucesso. Acurácia: 0.3400
